# Does Chunk Size Change What RAG Retrieves?

In RAG, documents are usually split into smaller chunks before creating embeddings.

I wanted to see how much the chunk size actually affects retrieval.

For this experiment, I'll use the same document, split it into different chunk sizes, and ask the same questions against each version.

In [1]:
import ollama
import numpy as np
import pandas as pd

## A small support document

I'll use one support document that contains information about login issues, password resets, account locks, file uploads, billing, and two-factor authentication.

Then I'll split the same document in different ways and compare what gets retrieved.

In [2]:
document = """
Account Access

Users who forget their password can reset it using the Forgot Password link on the login page.
A password reset link is valid for 15 minutes. If the link expires, the user needs to request a new one.

Accounts are temporarily locked after five failed login attempts.
A locked account automatically becomes available again after 30 minutes.
Users should avoid repeatedly attempting to sign in while the account is locked.

File Uploads

The application supports PDF, PNG, JPG, and CSV files.
Each uploaded file must be smaller than 10 MB.
If an upload fails, users should first check the file size and format before trying again.

Billing

Users can update their payment method from Settings > Billing > Payment Method.
Changes to the payment method apply to the next billing cycle.
Previous invoices remain available from the Billing History section.

Account Security

Two-factor authentication can be enabled from Settings > Security.
After enabling it, users must enter a verification code in addition to their password when signing in.
Recovery codes should be stored somewhere safe in case the authentication device is unavailable.
"""

print(document)


Account Access

Users who forget their password can reset it using the Forgot Password link on the login page.
A password reset link is valid for 15 minutes. If the link expires, the user needs to request a new one.

Accounts are temporarily locked after five failed login attempts.
A locked account automatically becomes available again after 30 minutes.
Users should avoid repeatedly attempting to sign in while the account is locked.

File Uploads

The application supports PDF, PNG, JPG, and CSV files.
Each uploaded file must be smaller than 10 MB.
If an upload fails, users should first check the file size and format before trying again.

Billing

Users can update their payment method from Settings > Billing > Payment Method.
Changes to the payment method apply to the next billing cycle.
Previous invoices remain available from the Billing History section.

Account Security

Two-factor authentication can be enabled from Settings > Security.
After enabling it, users must enter a verifica

## Creating different chunk sizes

I'll split the same document in three different ways.

Small chunks keep the information very focused. Medium chunks keep a few related sentences together. Large chunks keep an entire section together.

The content stays the same. Only the way it is split changes.

In [3]:
small_chunks = [
    "Users who forget their password can reset it using the Forgot Password link on the login page.",
    "A password reset link is valid for 15 minutes.",
    "If the link expires, the user needs to request a new one.",
    "Accounts are temporarily locked after five failed login attempts.",
    "A locked account automatically becomes available again after 30 minutes.",
    "Users should avoid repeatedly attempting to sign in while the account is locked.",

    "The application supports PDF, PNG, JPG, and CSV files.",
    "Each uploaded file must be smaller than 10 MB.",
    "If an upload fails, users should first check the file size and format before trying again.",

    "Users can update their payment method from Settings > Billing > Payment Method.",
    "Changes to the payment method apply to the next billing cycle.",
    "Previous invoices remain available from the Billing History section.",

    "Two-factor authentication can be enabled from Settings > Security.",
    "After enabling it, users must enter a verification code in addition to their password when signing in.",
    "Recovery codes should be stored somewhere safe in case the authentication device is unavailable."
]

In [4]:
medium_chunks = [
    """Users who forget their password can reset it using the Forgot Password link on the login page.
    A password reset link is valid for 15 minutes. If the link expires, the user needs to request a new one.""",

    """Accounts are temporarily locked after five failed login attempts.
    A locked account automatically becomes available again after 30 minutes.
    Users should avoid repeatedly attempting to sign in while the account is locked.""",

    """The application supports PDF, PNG, JPG, and CSV files.
    Each uploaded file must be smaller than 10 MB.
    If an upload fails, users should first check the file size and format before trying again.""",

    """Users can update their payment method from Settings > Billing > Payment Method.
    Changes to the payment method apply to the next billing cycle.
    Previous invoices remain available from the Billing History section.""",

    """Two-factor authentication can be enabled from Settings > Security.
    After enabling it, users must enter a verification code in addition to their password when signing in.
    Recovery codes should be stored somewhere safe in case the authentication device is unavailable."""
]

In [5]:
large_chunks = [
    """Account Access

    Users who forget their password can reset it using the Forgot Password link on the login page.
    A password reset link is valid for 15 minutes. If the link expires, the user needs to request a new one.

    Accounts are temporarily locked after five failed login attempts.
    A locked account automatically becomes available again after 30 minutes.
    Users should avoid repeatedly attempting to sign in while the account is locked.""",

    """File Uploads

    The application supports PDF, PNG, JPG, and CSV files.
    Each uploaded file must be smaller than 10 MB.
    If an upload fails, users should first check the file size and format before trying again.""",

    """Billing

    Users can update their payment method from Settings > Billing > Payment Method.
    Changes to the payment method apply to the next billing cycle.
    Previous invoices remain available from the Billing History section.""",

    """Account Security

    Two-factor authentication can be enabled from Settings > Security.
    After enabling it, users must enter a verification code in addition to their password when signing in.
    Recovery codes should be stored somewhere safe in case the authentication device is unavailable."""
]

In [6]:
print("Small chunks:", len(small_chunks))
print("Medium chunks:", len(medium_chunks))
print("Large chunks:", len(large_chunks))

Small chunks: 15
Medium chunks: 5
Large chunks: 4


In [7]:
def get_embeddings(texts):
    response = ollama.embed(
        model="nomic-embed-text",
        input=texts
    )

    return np.array(response["embeddings"])


def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )


def retrieve(query, chunks):
    chunk_embeddings = get_embeddings(chunks)
    query_embedding = get_embeddings([query])[0]

    similarities = [
        cosine_similarity(query_embedding, chunk_embedding)
        for chunk_embedding in chunk_embeddings
    ]

    best_index = int(np.argmax(similarities))

    return {
        "chunk": chunks[best_index],
        "similarity": similarities[best_index],
        "index": best_index
    }



In [8]:
query = "My account got locked after entering the wrong password. How long do I need to wait?"

print(query)

My account got locked after entering the wrong password. How long do I need to wait?


In [9]:
small_result = retrieve(query, small_chunks)
medium_result = retrieve(query, medium_chunks)
large_result = retrieve(query, large_chunks)

for name, result in [
    ("Small", small_result),
    ("Medium", medium_result),
    ("Large", large_result)
]:
    print(f"\n{name} chunks")
    print(f"Similarity: {result['similarity']:.3f}")
    print("Retrieved:")
    print(result["chunk"])


Small chunks
Similarity: 0.740
Retrieved:
A locked account automatically becomes available again after 30 minutes.

Medium chunks
Similarity: 0.781
Retrieved:
Accounts are temporarily locked after five failed login attempts.
    A locked account automatically becomes available again after 30 minutes.
    Users should avoid repeatedly attempting to sign in while the account is locked.

Large chunks
Similarity: 0.791
Retrieved:
Account Access

    Users who forget their password can reset it using the Forgot Password link on the login page.
    A password reset link is valid for 15 minutes. If the link expires, the user needs to request a new one.

    Accounts are temporarily locked after five failed login attempts.
    A locked account automatically becomes available again after 30 minutes.
    Users should avoid repeatedly attempting to sign in while the account is locked.


## Does chunk size change the final answer?

All three chunk sizes retrieved the information needed to answer the question.

But they returned different amounts of surrounding context.

I'll now give each retrieved chunk to the same LLM and ask the exact same question.

In [10]:
def generate_answer(query, context):
    prompt = f"""
You are answering a support question.

Use ONLY the information explicitly provided in the context below.
Do not use your own knowledge or make assumptions.

If the context does not contain enough information to answer any part
of the question, clearly say that the information is not available
in the provided context.

Context:
{context}

User question:
{query}

Keep the answer short and clear.
"""

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0
        }
    )

    return response["message"]["content"]

In [11]:
small_answer = generate_answer(
    query,
    small_result["chunk"]
)

medium_answer = generate_answer(
    query,
    medium_result["chunk"]
)

large_answer = generate_answer(
    query,
    large_result["chunk"]
)

for name, answer in [
    ("Small", small_answer),
    ("Medium", medium_answer),
    ("Large", large_answer)
]:
    print(f"\n{name} chunk answer:")
    print(answer)


Small chunk answer:
Your account will be automatically unlocked after 30 minutes.

Medium chunk answer:
According to the provided information, a locked account automatically becomes available again after 30 minutes.

Large chunk answer:
According to the provided context, a locked account automatically becomes available again after 30 minutes.


## A question that needs more than one fact

The first question only needed one piece of information, so every chunk size worked well.

Now I'll try a question that needs two related facts from the document and see what happens when those facts are split across chunks.

In [12]:
query_2 = """
My password reset link expired.
How long was it valid, and what should I do now?
""".strip()

print(query_2)

My password reset link expired.
How long was it valid, and what should I do now?


In [13]:
query_2 = """
My password reset link expired.
How long was it valid, and what should I do now?
""".strip()

print(query_2)

My password reset link expired.
How long was it valid, and what should I do now?


In [14]:
small_result_2 = retrieve(query_2, small_chunks)
medium_result_2 = retrieve(query_2, medium_chunks)
large_result_2 = retrieve(query_2, large_chunks)

for name, result in [
    ("Small", small_result_2),
    ("Medium", medium_result_2),
    ("Large", large_result_2)
]:
    print(f"\n{name} chunks")
    print(f"Similarity: {result['similarity']:.3f}")
    print("Retrieved:")
    print(result["chunk"])


Small chunks
Similarity: 0.801
Retrieved:
A password reset link is valid for 15 minutes.

Medium chunks
Similarity: 0.809
Retrieved:
Users who forget their password can reset it using the Forgot Password link on the login page.
    A password reset link is valid for 15 minutes. If the link expires, the user needs to request a new one.

Large chunks
Similarity: 0.768
Retrieved:
Account Access

    Users who forget their password can reset it using the Forgot Password link on the login page.
    A password reset link is valid for 15 minutes. If the link expires, the user needs to request a new one.

    Accounts are temporarily locked after five failed login attempts.
    A locked account automatically becomes available again after 30 minutes.
    Users should avoid repeatedly attempting to sign in while the account is locked.


In [15]:
small_answer_2 = generate_answer(
    query_2,
    small_result_2["chunk"]
)

medium_answer_2 = generate_answer(
    query_2,
    medium_result_2["chunk"]
)

large_answer_2 = generate_answer(
    query_2,
    large_result_2["chunk"]
)

for name, answer in [
    ("Small", small_answer_2),
    ("Medium", medium_answer_2),
    ("Large", large_answer_2)
]:
    print(f"\n{name} chunk answer:")
    print(answer)


Small chunk answer:
The password reset link was valid for 15 minutes. Since it has expired, you should try to request a new password reset link.

Medium chunk answer:
According to the provided context, the password reset link was valid for 15 minutes. Since the link has expired, you should request a new one.

Large chunk answer:
According to the provided information, a password reset link is valid for 15 minutes. Since the link has expired, you should request a new one.


## One more retrieval test

The previous question needed two related facts, and the smallest chunk didn't contain both of them.

I'll try one more question where the answer also depends on related information being kept together.

In [16]:
query_3 = """
I want to enable two-factor authentication.
Where do I enable it, and what will I need when I sign in afterward?
""".strip()

print(query_3)

I want to enable two-factor authentication.
Where do I enable it, and what will I need when I sign in afterward?


In [17]:
small_result_3 = retrieve(query_3, small_chunks)
medium_result_3 = retrieve(query_3, medium_chunks)
large_result_3 = retrieve(query_3, large_chunks)

for name, result in [
    ("Small", small_result_3),
    ("Medium", medium_result_3),
    ("Large", large_result_3)
]:
    print(f"\n{name} chunks")
    print(f"Similarity: {result['similarity']:.3f}")
    print("Retrieved:")
    print(result["chunk"])


Small chunks
Similarity: 0.850
Retrieved:
Two-factor authentication can be enabled from Settings > Security.

Medium chunks
Similarity: 0.845
Retrieved:
Two-factor authentication can be enabled from Settings > Security.
    After enabling it, users must enter a verification code in addition to their password when signing in.
    Recovery codes should be stored somewhere safe in case the authentication device is unavailable.

Large chunks
Similarity: 0.803
Retrieved:
Account Security

    Two-factor authentication can be enabled from Settings > Security.
    After enabling it, users must enter a verification code in addition to their password when signing in.
    Recovery codes should be stored somewhere safe in case the authentication device is unavailable.


In [18]:
small_answer_3 = generate_answer(
    query_3,
    small_result_3["chunk"]
)

medium_answer_3 = generate_answer(
    query_3,
    medium_result_3["chunk"]
)

large_answer_3 = generate_answer(
    query_3,
    large_result_3["chunk"]
)

for name, answer in [
    ("Small", small_answer_3),
    ("Medium", medium_answer_3),
    ("Large", large_answer_3)
]:
    print(f"\n{name} chunk answer:")
    print(answer)


Small chunk answer:
To enable two-factor authentication, go to Settings > Security.

When signing in afterward, you will need a second form of verification, in addition to your password.

Medium chunk answer:
To enable two-factor authentication, go to Settings > Security. When signing in, you will need to enter a verification code in addition to your password.

Large chunk answer:
To enable two-factor authentication, go to Settings > Security. When signing in, you will need to enter a verification code in addition to your password.


## Comparing the three tests

After trying three different questions, I noticed that similarity score alone didn't tell the full story.

| Question | Small chunks | Medium chunks | Large chunks |
|---|---|---|---|
| Account unlock | Complete and focused | Complete with useful context | Complete with extra context |
| Password reset | Missing one related fact | Complete and focused | Complete with extra context |
| Two-factor authentication | Missing one related fact | Complete and focused | Complete with extra context |

Small chunks were very focused, but sometimes split information that was needed together.

Large chunks usually had enough information, but also brought in details that weren't needed for the question.

For these examples, the medium chunks gave a better balance between keeping related information together and avoiding unrelated context.

## What I found

I initially thought smaller chunks might work better because they are more focused.

But that wasn't always enough.

For two of the questions, the most similar small chunk didn't contain all the information needed to answer the question. The LLM still produced a reasonable answer, but part of it came from what the model already knew rather than from the retrieved context.

Medium chunks kept the related facts together in these examples.

Large chunks also had the required information, but sometimes included extra details that weren't relevant to the question.

In [19]:
def get_embeddings(texts):
    response = ollama.embed(
        model="nomic-embed-text",
        input=texts
    )

    return np.array(response["embeddings"])


def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )


def retrieve(query, chunks):
    chunk_embeddings = get_embeddings(chunks)
    query_embedding = get_embeddings([query])[0]

    similarities = [
        cosine_similarity(query_embedding, chunk_embedding)
        for chunk_embedding in chunk_embeddings
    ]

    best_index = int(np.argmax(similarities))

    return {
        "chunk": chunks[best_index],
        "similarity": similarities[best_index],
        "index": best_index
    }



## Takeaway

Chunk size isn't just about getting a higher similarity score.

The retrieved chunk also needs to contain enough context to answer the question.

Too small can separate related information. Too large can bring unnecessary information along.

For me, the useful question isn't "What chunk size is best?"

It's:

**Does the retrieved chunk contain the right amount of context for the question?**